<a href="https://colab.research.google.com/github/athitthiyan/Learning_Gen_AI/blob/main/Autogen_groq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install AutoGen Studio and pyngrok
!pip install -q autogenstudio pyngrok

# Verify the installation
!autogenstudio version

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.7/91.7 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.4/83.4 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.1/106.1 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.6/96.6 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.9/296.9 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [2]:
import os
import getpass

# 1. Set up Groq API Key
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

# 2. Set up Ngrok Auth Token (Required to tunnel the AutoGen Studio UI)
NGROK_TOKEN = getpass.getpass("Enter your Ngrok Auth Token: ")
!ngrok config add-authtoken {NGROK_TOKEN}

Enter your Groq API Key: ··········
Enter your Ngrok Auth Token: ··········
Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [3]:
import multiprocessing
import time
from pyngrok import ngrok

def run_autogen_studio():
    # Launch AutoGen Studio on port 8081
    !autogenstudio ui --port 8081 --host 0.0.0.0

# Start AutoGen Studio in a background process
process = multiprocessing.Process(target=run_autogen_studio)
process.start()

# Wait a moment for the server to spin up
time.sleep(5)

# Open the ngrok tunnel to port 8081
public_url = ngrok.connect(8081)
print("\n" + "="*60)
print(f"[SUCCESS] AutoGen Studio is running!")
print(f"Click the link below to open the UI:")
print(f"{public_url}")
print("="*60 + "\n")


[SUCCESS] AutoGen Studio is running!
Click the link below to open the UI:
NgrokTunnel: "https://demisable-tressa-excessive.ngrok-free.dev" -> "http://localhost:8081"



In [14]:
import os
import asyncio
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import MaxMessageTermination
from autogen_ext.models.openai import OpenAIChatCompletionClient
from google.colab import userdata
from autogen_agentchat.base import TaskResult

async def run_team():
    # Fetch the Groq API key you securely entered in Cell 2

    groq_key = userdata.get("GROQ_API_KEY")
    # groq_key = ""

    # Define Groq model clients with the correct model_info settings
    researcher_model = OpenAIChatCompletionClient(
        model="llama-3.1-8b-instant",
        base_url="https://api.groq.com/openai/v1",
        api_key=groq_key,
        model_info={
            "vision": False,
            "function_calling": True,
            "json_output": True,
            "structured_output": True,
            "family": "unknown"
        }
    )

    editor_model = OpenAIChatCompletionClient(
        model="llama-3.3-70b-versatile",
        base_url="https://api.groq.com/openai/v1",
        api_key=groq_key,
        model_info={
            "vision": False,
            "function_calling": True,
            "json_output": True,
            "structured_output": True,
            "family": "unknown"
        }
    )

    # Define the individual Agents
    researcher = AssistantAgent(
        name="Researcher",
        model_client=researcher_model,
        system_message="You are an expert researcher. Provide a highly detailed summary using clear Markdown formatting."
    )

    editor = AssistantAgent(
        name="Editor",
        model_client=editor_model,
        system_message="You are a strict editor. Critique the researcher's work and optimize it for professional delivery."
    )

    # Orchestrate the workflow team
    team = RoundRobinGroupChat(
        participants=[researcher, editor],
        termination_condition=MaxMessageTermination(max_messages=4)
    )

    # Run the prompt
    print("--- Starting Multi-Agent Session ---")
    # async for message in team.run_stream(task="Explain why Groq LPUs provide higher throughput for LLMs than standard GPUs."):
    #   print(f"\n\033[1m[{message.source}]\033[0m: {message.content}")
    #   print("-" * 40)

    async for event in team.run_stream(task="Explain why Groq LPUs provide higher throughput for LLMs than standard GPUs."):
      if isinstance(event, TaskResult):
        print("\n=== Task Finished ===")
        print(event)
      else:
        print(f"\n[{event.source}]")
        print(event.content)
        print("-" * 40)

# Execute the async loop inside Google Colab
await run_team()

--- Starting Multi-Agent Session ---

[user]
Explain why Groq LPUs provide higher throughput for LLMs than standard GPUs.
----------------------------------------

[Researcher]
**Overview of Groq LPUs and their Advantages for LLMs**

### Introduction

Groq is a Silicon Valley-based AI chip startup that has recently introduced a new type of AI chip called the Lookout Processing Unit (LPU). The Groq LPU has shown significant advantages over standard Graphics Processing Units (GPUs) for Large Language Models (LLMs). In this summary, we will explore the reasons why Groq LPUs provide higher throughput for LLMs than standard GPUs.

### Limitations of Standard GPUs for LLMs
--------------------------------------

Standard GPUs are not designed for sequential processing, which is a critical requirement for LLMs. LLMs require processing long sequences of text, making them less suitable for the parallel processing architecture of GPUs. This limitation leads to several issues:

#### 1. Data Trans